<a href="https://colab.research.google.com/github/vivek09thakur/XDEA/blob/main/getting-started-with-quantam-computing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classical vs Quantum Computing — side by side

This notebook shows the **difference between a normal (classical) computer and a quantum computer** using tiny programs you can run and compare.

Every section follows the same pattern:

1. A short explanation in plain language.
2. A **classical program** written in plain Python.
3. The **same task as a quantum program** written with [Cirq](https://quantumai.google/cirq) (a quantum circuit library from Google).
4. Look at the outputs and spot the difference.

No prior quantum knowledge is needed. The quantum circuits run on a *simulator* on your own machine, so no real quantum hardware is required.

## The one-line summary

| | Classical computer | Quantum computer |
|---|---|---|
| Basic unit | **bit** — exactly `0` or `1` | **qubit** — can be `0`, `1`, or a *blend* of both (superposition) |
| State of 2 units | one of `00`, `01`, `10`, `11` | a weighted mix of **all four** at once |
| Reading a value | always the same answer | *measurement* picks one answer at random, based on the weights |
| Link between units | none — bits are independent | qubits can be **entangled**: measuring one instantly fixes the other |
| Scaling | $n$ bits store $n$ values | $n$ qubits describe $2^n$ amplitudes at once |

Let's see each of these rows as code.

## Setup

The classical examples only need plain Python. The quantum examples need `cirq`, which the next cell installs if it's missing.

In [ ]:
import random
from collections import Counter

try:
    import cirq
except ImportError:
    %pip install cirq --quiet
    import cirq

print("cirq version:", cirq.__version__)

## 1. A bit vs a qubit

**Classical:** a bit is a variable that is `0` or `1`. Read it 10 times and you get the same answer 10 times.

**Quantum:** a qubit starts at `0`. Apply a *Hadamard gate* (`H`) and it becomes an equal blend of `0` and `1`. Every time you measure it, it randomly collapses to `0` or `1` — about 50/50.

### Classical bit

In [ ]:
bit = 0

readings = [bit for _ in range(10)]
print("Classical bit read 10 times:", readings)
print("Counts:", dict(Counter(readings)))

In [ ]:
q = cirq.LineQubit(0)

circuit = cirq.Circuit(
    cirq.H(q),
    cirq.measure(q, key="m"),
)
print("Quantum circuit:")
print(circuit)
print()

result = cirq.Simulator().run(circuit, repetitions=10)
readings = result.measurements["m"].flatten().tolist()
print("Qubit read 10 times:", readings)
print("Counts:", dict(Counter(readings)))

**What to notice:** the classical bit gave the same value every time. The qubit gave a mix of `0`s and `1`s from the *same* circuit — the randomness is built into the physics, not into the code.

> "But I can get random bits in Python with `random.randint(0, 1)`!" — true, and that's a fair objection. The difference is *where* the randomness comes from and what else the qubit can do, which the next sections show.

---

## 2. Two bits vs two qubits: how much is "in" the state?

**Classical:** two bits hold exactly **one** of four values at a time.

**Quantum:** two qubits in superposition hold **all four** values at once, each with an *amplitude*. Cirq can print the whole internal state vector of the simulator so you can see this directly.

### Classical: two bits

In [ ]:
bit_a, bit_b = 1, 0

state = f"{bit_a}{bit_b}"
print("Two classical bits hold exactly one value:", state)
print("Possible values they COULD hold: 00, 01, 10, 11 — but only one at a time.")

In [ ]:
q0, q1 = cirq.LineQubit.range(2)

circuit = cirq.Circuit(cirq.H(q0), cirq.H(q1))
print("Quantum circuit:")
print(circuit)
print()

state = cirq.Simulator().simulate(circuit)
print("Full quantum state (all 4 values at once, each with amplitude 0.5):")
print(state.dirac_notation())
print()

print("Probability of each outcome if we measure:")
for i, amp in enumerate(state.final_state_vector):
    print(f"  |{i:02b}>  amplitude = {amp.real:+.2f}   probability = {abs(amp)**2:.2f}")

**What to notice:** the classical pair holds one string. The quantum pair holds a *vector* with an entry for every string. With $n$ qubits that vector has $2^n$ entries — 300 qubits would need more amplitudes than there are atoms in the universe, which is why quantum simulators on normal computers hit a wall at around 40–50 qubits.

---

## 3. Entanglement: a link that classical bits don't have

**Classical:** if you set two bits randomly and independently, knowing one tells you nothing about the other. You'll see all four combinations `00, 01, 10, 11`.

**Quantum:** `H` on the first qubit, then a `CNOT` (controlled-NOT) linking it to the second, produces a *Bell state*. The two qubits are random individually, but **always agree**. You will only ever see `00` or `11` — never `01` or `10`.

### Classical: two independent random bits

In [ ]:
random.seed(0)
shots = 20

pairs = [f"{random.randint(0, 1)}{random.randint(0, 1)}" for _ in range(shots)]
print("Classical random pairs:", " ".join(pairs))
print("Counts:", dict(sorted(Counter(pairs).items())))
print("Pairs that DISAGREE (01 or 10):", sum(p in ("01", "10") for p in pairs), "of", shots)

### Quantum: two entangled qubits

In [ ]:
q0, q1 = cirq.LineQubit.range(2)
shots = 20

circuit = cirq.Circuit(
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key="m"),
)
print("Quantum circuit:")
print(circuit)
print()

result = cirq.Simulator().run(circuit, repetitions=shots)
pairs = ["".join(str(b) for b in row) for row in result.measurements["m"]]
print("Quantum entangled pairs:", " ".join(pairs))
print("Counts:", dict(sorted(Counter(pairs).items())))
print("Pairs that DISAGREE (01 or 10):", sum(p in ("01", "10") for p in pairs), "of", shots)

**What to notice:** the classical pairs disagree about half the time. The entangled qubits **never** disagree, even though each one on its own is a 50/50 coin flip. There is no classical program that produces this behaviour without one bit secretly "looking at" the other — in the quantum circuit, nothing is copied after the measurement; the correlation comes from the shared state.

---

## 4. A real speed-up: finding the one marked item (Grover's search)

Here's a problem where the quantum approach needs *fewer steps*, not just "different" behaviour.

**Task:** there are 4 boxes (`00`, `01`, `10`, `11`). Exactly one is marked. You have an `is_marked(box)` function (an *oracle*) but you can't see inside it. Find the marked box with as few oracle calls as possible.

- **Classical:** you must open boxes one at a time. Worst case **3 checks** (if the first three are empty, the fourth must be it); on average about 2.
- **Quantum (Grover's algorithm):** put all 4 boxes in superposition, call the oracle **once**, and the marked box's probability jumps to 100%.

### Classical search

In [ ]:
MARKED = "11"


def is_marked(box: str) -> bool:
    return box == MARKED


def classical_search(boxes):
    oracle_calls = 0
    for box in boxes:
        oracle_calls += 1
        if is_marked(box):
            return box, oracle_calls
    raise ValueError("no marked box")


boxes = ["00", "01", "10", "11"]
found, calls = classical_search(boxes)
print(f"Classical: found {found} after {calls} oracle call(s)")
print("Worst case for 4 boxes: 3 calls.  For N boxes: about N/2 on average.")

### Quantum search (Grover, 2 qubits)

The oracle below flips the sign of the amplitude on the marked box. Because every box is in superposition, **one** oracle call touches all four at once. A "diffusion" step then amplifies the flipped one.

In [ ]:
q0, q1 = cirq.LineQubit.range(2)


def oracle(marked: str):
    ops = []
    for qubit, bit in zip((q0, q1), marked):
        if bit == "0":
            ops.append(cirq.X(qubit))
    ops.append(cirq.CZ(q0, q1))
    for qubit, bit in zip((q0, q1), marked):
        if bit == "0":
            ops.append(cirq.X(qubit))
    return ops


def diffusion():
    return [
        cirq.H(q0), cirq.H(q1),
        cirq.X(q0), cirq.X(q1),
        cirq.CZ(q0, q1),
        cirq.X(q0), cirq.X(q1),
        cirq.H(q0), cirq.H(q1),
    ]


circuit = cirq.Circuit(
    cirq.H(q0), cirq.H(q1),
    oracle(MARKED),
    diffusion(),
    cirq.measure(q0, q1, key="m"),
)
print("Quantum circuit:")
print(circuit)
print()

result = cirq.Simulator().run(circuit, repetitions=20)
answers = ["".join(str(b) for b in row) for row in result.measurements["m"]]
print("Measured box over 20 runs:", dict(Counter(answers)))
print(f"Quantum: found {Counter(answers).most_common(1)[0][0]} after 1 oracle call")
print("For N boxes Grover needs about sqrt(N) calls instead of N/2.")

**What to notice:** every one of the 20 runs lands on the marked box after a *single* oracle call, where the classical loop needed up to 3. Change `MARKED` to `"00"`, `"01"` or `"10"` and re-run both search cells — the quantum version still finds it in one call.

---

## 5. Where quantum is *not* better

Quantum computers are not "faster computers". They're faster at a *few specific kinds* of problems (searching unstructured data, factoring large numbers, simulating molecules). For ordinary tasks they're slower, and simulating even a few dozen qubits on a classical machine is already very expensive, as the state-vector size $2^n$ shows below.

In [ ]:
print(f"{'qubits':>6} | {'amplitudes to track (2^n)':>26} | {'memory as complex64':>20}")
print("-" * 60)
for n in (2, 10, 20, 30, 40, 50):
    amplitudes = 2**n
    gigabytes = amplitudes * 8 / 1e9
    print(f"{n:>6} | {amplitudes:>26,} | {gigabytes:>17.3g} GB")

## Recap

| Experiment | Classical result | Quantum result |
|---|---|---|
| Read one unit 10× | same value every time | random `0`/`1` mix from one fixed circuit |
| State of two units | one string, e.g. `10` | a vector over all four strings |
| Two random units | `00, 01, 10, 11` all appear | only `00` and `11` — perfectly correlated |
| Find 1 of 4 boxes | up to 3 oracle calls | 1 oracle call |

**Superposition** (many values at once), **entanglement** (linked qubits), and **interference** (amplifying the right answer, as in Grover) are the three things a classical bit simply cannot do — and they're the whole reason quantum computers are interesting.

### Try it yourself
- In section 1, replace `cirq.H(q)` with `cirq.X(q)`. The qubit becomes a plain `1` — a qubit *can* act like a normal bit.
- In section 3, remove the `cirq.CNOT` line. The disagreements come back.
- In section 4, change `MARKED` and re-run both search cells.